# Step 4 — Hybrid Search & Retrieval UI
Two search modes:
1. **Hybrid search** — dense (semantic) + sparse (BM25) fused with RRF  
2. **Filtered retrieval** — by `year`, `naics3`, `section`, `ticker`  

Interactive widget UI built with `ipywidgets`.

In [ ]:
!pip install -q milvus-lite 'pymilvus>=2.4.0' sentence-transformers rank-bm25 ipywidgets

from google.colab import drive
drive.mount('/content/drive')

import os, json, pickle
import numpy as np

BASE_DIR   = '/content/drive/MyDrive/mdna_rag'
SPARSE_IN  = f'{BASE_DIR}/embeddings_sparse.pkl'
MILVUS_DB  = f'{BASE_DIR}/milvus_mdna.db'
COLLECTION = 'mdna_2024_25'

print('✅ Setup complete')

In [ ]:
# ── Load models & index ───────────────────────────────────────────────────────
from pymilvus import MilvusClient, AnnSearchRequest, RRFRanker
from sentence_transformers import SentenceTransformer
import re, pickle
from collections import defaultdict

client = MilvusClient(MILVUS_DB)
client.load_collection(COLLECTION)
print('✅ Milvus collection loaded')

dense_model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')
print('✅ Dense model loaded')

with open(SPARSE_IN, 'rb') as f:
    sp_data = pickle.load(f)
VOCAB: dict[str, int] = sp_data['vocab']
print(f'✅ BM25 vocab loaded ({len(VOCAB):,} tokens)')

In [ ]:
# ── Encoding helpers ──────────────────────────────────────────────────────────
STOPWORDS = {
    'the','a','an','and','or','but','in','on','at','to','for','of','with',
    'as','by','from','is','was','are','were','be','this','that','have',
    'had','has','will','would','could','should','may','also','such',
    'than','more','other','any','all','been','it','its','we','our','us',
}

def tokenize(text: str) -> list[str]:
    tokens = re.findall(r'[A-Za-z0-9][A-Za-z0-9\-\.]*', text.lower())
    return [t for t in tokens if t not in STOPWORDS and len(t) > 1]

def encode_dense(query: str) -> list[float]:
    vec = dense_model.encode(query, normalize_embeddings=True)
    return vec.tolist()

# Approximate IDF from vocab size (use uniform weight if IDF not persisted)
import math
N_DOCS = 150_000   # approximate — update if you know your corpus size

def encode_sparse(query: str) -> dict[int, float]:
    tokens = tokenize(query)
    freq: dict[str, int] = defaultdict(int)
    for t in tokens:
        freq[t] += 1
    sparse = {}
    for tok, tf in freq.items():
        if tok in VOCAB:
            # simple TF weight; replace with full BM25 if IDF is persisted
            sparse[VOCAB[tok]] = float(tf)
    return sparse

print('✅ Encoding helpers ready')

In [ ]:
# ── Core search function ──────────────────────────────────────────────────────
OUTPUT_FIELDS = [
    'ticker','year','naics3','naics_label','section',
    'chunk_index','token_count','text','accession'
]

def build_filter(year=None, naics3=None, ticker=None, section=None) -> str:
    """Build Milvus boolean filter expression from optional params."""
    clauses = []
    if year:
        if isinstance(year, (list, tuple)):
            clauses.append(f'year in {list(year)}')
        else:
            clauses.append(f'year == {int(year)}')
    if naics3:
        if isinstance(naics3, (list, tuple)):
            clauses.append(f'naics3 in {list(naics3)}')
        else:
            clauses.append(f'naics3 == {int(naics3)}')
    if ticker:
        tickers = [ticker.upper()] if isinstance(ticker, str) else [t.upper() for t in ticker]
        clauses.append(f'ticker in {tickers}')
    if section:
        sections = [section] if isinstance(section, str) else list(section)
        clauses.append(f'section in {sections}')
    return ' and '.join(clauses) if clauses else ''


def hybrid_search(
    query       : str,
    year        = None,       # int, list[int], or None
    naics3      = None,       # int, list[int], or None
    ticker      = None,       # str, list[str], or None
    section     = None,       # str, list[str], or None
    top_k       : int = 5,
    dense_k     : int = 20,
    sparse_k    : int = 20,
    rrf_k       : int = 60,   # RRF smoothing constant
) -> list[dict]:
    """
    Hybrid search: dense cosine + sparse BM25, fused with Reciprocal Rank Fusion.
    All filters applied server-side before vector search.
    """
    filter_expr  = build_filter(year=year, naics3=naics3, ticker=ticker, section=section)
    dense_vec    = encode_dense(query)
    sparse_vec   = encode_sparse(query)

    dense_req = AnnSearchRequest(
        data        = [dense_vec],
        anns_field  = 'dense_vec',
        param       = {'metric_type': 'COSINE', 'params': {'ef': 100}},
        limit       = dense_k,
        expr        = filter_expr or None,
    )
    sparse_req = AnnSearchRequest(
        data        = [sparse_vec],
        anns_field  = 'sparse_vec',
        param       = {'metric_type': 'IP', 'params': {}},
        limit       = sparse_k,
        expr        = filter_expr or None,
    )

    results = client.hybrid_search(
        collection_name = COLLECTION,
        reqs            = [dense_req, sparse_req],
        ranker          = RRFRanker(k=rrf_k),
        limit           = top_k,
        output_fields   = OUTPUT_FIELDS,
    )

    hits = []
    for hit in results[0]:
        entity = hit.get('entity', hit)   # pymilvus ≥2.4 returns entity dict
        hits.append({
            'score'       : round(hit.get('distance', 0), 4),
            'ticker'      : entity.get('ticker'),
            'year'        : entity.get('year'),
            'naics3'      : entity.get('naics3'),
            'naics_label' : entity.get('naics_label'),
            'section'     : entity.get('section'),
            'text'        : entity.get('text'),
            'accession'   : entity.get('accession'),
        })
    return hits


def filter_retrieve(
    year    = None,
    naics3  = None,
    ticker  = None,
    section = None,
    limit   : int = 20,
) -> list[dict]:
    """Pure metadata retrieval — no vector search, just filter."""
    filter_expr = build_filter(year=year, naics3=naics3, ticker=ticker, section=section)
    if not filter_expr:
        raise ValueError('Provide at least one filter for filter_retrieve()')
    return client.query(
        collection_name = COLLECTION,
        filter          = filter_expr,
        output_fields   = OUTPUT_FIELDS,
        limit           = limit,
    )

print('✅ Search functions ready')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# INTERACTIVE UI  (ipywidgets)
# ─────────────────────────────────────────────────────────────────────────────
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# ── Common NAICS 3-digit industries in SEC filings ────────────────────────────
NAICS_OPTIONS = [
    ('All industries', 0),
    ('211 — Oil & Gas Extraction', 211),
    ('212 — Mining', 212),
    ('221 — Utilities', 221),
    ('236 — Construction', 236),
    ('311 — Food Manufacturing', 311),
    ('325 — Chemical / Pharma Mfg', 325),
    ('334 — Computer & Electronic', 334),
    ('336 — Transportation Equip', 336),
    ('339 — Medical Devices', 339),
    ('423 — Wholesale Durable', 423),
    ('441 — Motor Vehicle Retail', 441),
    ('452 — General Merchandise', 452),
    ('481 — Air Transportation', 481),
    ('484 — Truck Transportation', 484),
    ('511 — Software Publishers', 511),
    ('517 — Telecom', 517),
    ('519 — Internet & Data Svcs', 519),
    ('521 — Banking', 521),
    ('523 — Securities & Investment', 523),
    ('524 — Insurance', 524),
    ('531 — Real Estate', 531),
    ('541 — Professional Services', 541),
    ('621 — Healthcare', 621),
    ('711 — Entertainment', 711),
    ('721 — Accommodation', 721),
    ('722 — Food Services', 722),
]

SECTION_OPTIONS = [
    'All sections',
    'overview',
    'results_of_operations',
    'revenue',
    'gross_profit',
    'operating_expenses',
    'operating_income',
    'net_income',
    'segment_results',
    'liquidity',
    'cash_flows',
    'critical_accounting',
    'off_balance_sheet',
    'contractual_obligations',
    'market_risk',
    'outlook',
]

# ── Widget definitions ────────────────────────────────────────────────────────
style   = {'description_width': '140px'}
layout  = widgets.Layout(width='600px')
layout2 = widgets.Layout(width='300px')

query_box   = widgets.Textarea(
    value       = 'What drove revenue growth in cloud services?',
    description = 'Query:',
    layout      = widgets.Layout(width='600px', height='70px'),
    style       = style,
)
year_sel    = widgets.SelectMultiple(
    options     = [2024, 2025],
    value       = [2024, 2025],
    description = 'Filing year(s):',
    layout      = widgets.Layout(width='300px', height='60px'),
    style       = style,
)
naics_sel   = widgets.Dropdown(
    options     = NAICS_OPTIONS,
    value       = 0,
    description = 'Industry (NAICS3):',
    layout      = layout2,
    style       = style,
)
section_sel = widgets.Dropdown(
    options     = SECTION_OPTIONS,
    value       = 'All sections',
    description = 'MD&A section:',
    layout      = layout2,
    style       = style,
)
ticker_box  = widgets.Text(
    value       = '',
    placeholder = 'e.g. AAPL, MSFT (comma-sep, optional)',
    description = 'Ticker(s):',
    layout      = layout2,
    style       = style,
)
topk_slider = widgets.IntSlider(
    value=5, min=1, max=20, step=1,
    description='Top-K results:',
    style=style, layout=layout2,
)
mode_toggle = widgets.ToggleButtons(
    options     = ['Hybrid Search', 'Filter Only'],
    value       = 'Hybrid Search',
    description = 'Mode:',
    style       = {'description_width': '60px',
                   'button_width': '160px'},
)
run_btn     = widgets.Button(
    description = '🔍  Search',
    button_style= 'primary',
    layout      = widgets.Layout(width='200px', height='40px'),
)
out_panel   = widgets.Output()

# ── Layout ────────────────────────────────────────────────────────────────────
filters_row  = widgets.HBox([year_sel, naics_sel, section_sel])
options_row  = widgets.HBox([ticker_box, topk_slider, mode_toggle])
ui = widgets.VBox([
    widgets.HTML('<h3 style="font-family:monospace;margin-bottom:4px">'
                 '📄 SEC MD&A Hybrid Search — 2024/25</h3>'),
    query_box,
    filters_row,
    options_row,
    run_btn,
    out_panel,
])

# ── Result renderer ───────────────────────────────────────────────────────────
def render_hit(hit: dict, rank: int) -> str:
    score_tag = f"<code style='color:#2a9d8f'>RRF={hit['score']}</code>" \
                if 'score' in hit else ''
    text_preview = (hit.get('text') or '')[:400].replace('<','&lt;')
    return f"""
    <div style='border:1px solid #ddd;border-radius:6px;padding:12px;
                margin-bottom:10px;font-family:monospace;font-size:13px'>
      <div style='margin-bottom:6px'>
        <b>#{rank}</b>
        <span style='background:#264653;color:#fff;padding:2px 8px;
                     border-radius:4px;margin:0 4px'>{hit.get('ticker','?')}</span>
        <span style='background:#e9c46a;color:#000;padding:2px 8px;
                     border-radius:4px;margin:0 4px'>{hit.get('year','?')}</span>
        <span style='background:#f4a261;color:#fff;padding:2px 8px;
                     border-radius:4px;margin:0 4px'>
          NAICS {hit.get('naics3','?')}</span>
        <span style='background:#a8dadc;color:#1d3557;padding:2px 8px;
                     border-radius:4px;margin:0 4px'>{hit.get('section','?')}</span>
        {score_tag}
      </div>
      <div style='color:#555;font-size:11px;margin-bottom:6px'>
        {hit.get('naics_label','')} | accession: {hit.get('accession','')[:20]}...
      </div>
      <div style='line-height:1.6'>{text_preview}{'...' if len(hit.get('text',''))>400 else ''}</div>
    </div>"""


def on_search(btn):
    out_panel.clear_output()
    with out_panel:
        q       = query_box.value.strip()
        years   = list(year_sel.value)
        naics_v = naics_sel.value
        sec_v   = section_sel.value
        tk_raw  = ticker_box.value.strip()
        topk    = topk_slider.value
        mode    = mode_toggle.value

        naics_arg   = naics_v  if naics_v   else None
        section_arg = sec_v    if sec_v != 'All sections' else None
        ticker_arg  = [t.strip().upper() for t in tk_raw.split(',') if t.strip()] or None

        try:
            if mode == 'Hybrid Search':
                if not q:
                    display(HTML('<p style="color:red">Enter a query.</p>'))
                    return
                hits = hybrid_search(
                    query   = q,
                    year    = years,
                    naics3  = naics_arg,
                    ticker  = ticker_arg,
                    section = section_arg,
                    top_k   = topk,
                )
                header = f'<b>Hybrid results for:</b> "{q}"'
            else:
                hits = filter_retrieve(
                    year    = years,
                    naics3  = naics_arg,
                    ticker  = ticker_arg,
                    section = section_arg,
                    limit   = topk,
                )
                header = '<b>Filter-only results</b>'

            if not hits:
                display(HTML('<p style="color:orange">No results found. '
                             'Try broadening filters.</p>'))
                return

            html_parts = [f'<div style="margin-bottom:8px">{header} '
                          f'— <code>{len(hits)}</code> hits</div>']
            for i, hit in enumerate(hits, 1):
                html_parts.append(render_hit(hit, i))
            display(HTML(''.join(html_parts)))

        except Exception as e:
            display(HTML(f'<p style="color:red">Error: {e}</p>'))

run_btn.on_click(on_search)
display(ui)

---
## Programmatic usage examples
Run these cells directly without the UI.

In [ ]:
# ── Example 1: Hybrid search — revenue growth in tech, 2024 ──────────────────
results = hybrid_search(
    query   = 'What drove cloud revenue growth?',
    year    = 2024,
    naics3  = 519,             # Internet & Data Services
    section = 'revenue',
    top_k   = 5,
)
for r in results:
    print(f"[{r['ticker']} {r['year']} | {r['section']}] score={r['score']}")
    print(f"  {r['text'][:150]}...\n")

In [ ]:
# ── Example 2: Hybrid search — liquidity across pharma, 2024 & 2025 ──────────
results = hybrid_search(
    query   = 'liquidity risk and debt maturity concerns',
    year    = [2024, 2025],
    naics3  = 325,             # Chemical / Pharma
    section = 'liquidity',
    top_k   = 5,
)
for r in results:
    print(f"[{r['ticker']} {r['year']}] score={r['score']}")
    print(f"  {r['text'][:150]}...\n")

In [ ]:
# ── Example 3: Filter only — all AAPL chunks from 2024 ───────────────────────
results = filter_retrieve(
    ticker  = 'AAPL',
    year    = 2024,
    limit   = 10,
)
for r in results:
    print(f"  [{r['section']}] {r['text'][:120]}...")

In [ ]:
# ── Example 4: Cross-industry comparison — operating margins ─────────────────
for naics_code, label in [(334, 'Semiconductors'), (721, 'Hotels'), (481, 'Airlines')]:
    hits = hybrid_search(
        query   = 'operating margin pressure cost inflation',
        year    = [2024, 2025],
        naics3  = naics_code,
        section = 'results_of_operations',
        top_k   = 2,
    )
    print(f'\n── {label} (NAICS {naics_code}) ──')
    for h in hits:
        print(f"  [{h['ticker']} {h['year']}] {h['text'][:140]}...")